In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings

# ----------------------
# Config / paths
# ----------------------
folder = Path(r"C:\Users\ageglio\OneDrive - TRC\Documents\My EQuIS Work\Mdn01\Mosaic Hutchinson")

results_path = folder / "Analytical Results II Crosstab.xlsm"
coord_path   = folder / "dt_coordinate.xlsx"
seg_path     = folder / "dt_well_segment.xlsx"

# no filter option (use all wells with both coords and results):
active_csv   = "MosaicWellNames.csv"  # CSV with one column of well IDs (no header)
save_coord_csv = "MosaicWellCoordinates_all_sampled.csv"  # Optional: save cleaned coordinates for use in GWSDAT or other tools
save_monitored_csv = "MosaicMonitoringResults_all_sampled.csv"  # Optional: save list of monitored wells for reference

# filtered option (only wells that are in the active list and have monitoring data):
# active_csv = "MosaicWellNames_Filtered.csv"  # Use the filtered list if you want to intersect with the plotted wells
# save_coord_csv = "MosaicWellCoordinates_filtered.csv"  # Optional: save cleaned coordinates for use in GWSDAT or other tools
# save_monitored_csv = "MosaicMonitoringResults_filtered.csv"  # Optional: save list of monitored wells for reference

# ----------------------
# Helpers: well ID normalization & safe date parsing
# ----------------------
DASHES = {
    "\u2010",  # hyphen
    "\u2011",  # non-breaking hyphen
    "\u2012",  # figure dash
    "\u2013",  # en dash
    "\u2014",  # em dash
    "\u2015",  # horizontal bar
    "\u2212",  # minus sign
    "\u2043",  # hyphen bullet
}
dash_pattern = re.compile("|".join(map(re.escape, DASHES)))
zero_width_pattern = re.compile(r"[\u200B-\u200D\uFEFF]")  # ZW space/joiners

def normalize_well_id(x) -> str:
    """Normalize well identifier for consistent matching across sources."""
    if pd.isna(x):
        return np.nan
    s = str(x)
    s = unicodedata.normalize("NFKC", s)          # canonical/compat normalization
    s = s.replace("\u00A0", " ")                  # non-breaking space -> space
    s = zero_width_pattern.sub("", s)             # drop zero-width chars
    s = dash_pattern.sub("-", s)                  # unify all dash-like chars to '-'
    s = re.sub(r"\s+", " ", s).strip()            # collapse whitespace
    s = s.upper()                                 # enforce consistent case
    return s

def to_gwsdat_date(series: pd.Series) -> pd.Series:
    """Parse dates and format as YYYY-mm-dd for GWSDAT."""
    dt = pd.to_datetime(series, errors="coerce")
    return dt.dt.strftime("%Y-%m-%d")

# ----------------------
# 1) Read + process results data
# ----------------------
df_results = pd.read_excel(
    results_path,
    sheet_name="General Chemistry",
    skiprows=1,
    engine="openpyxl"
)

# Standardize expected columns (adjust here if source columns vary)
rename_results = {
    "SYS_LOC_CODE": "WellName",
    "Analyte": "Constituent",
    "SAMPLE_DATE": "RealSampleDate",
    "RESULT_NUMERIC": "Result",
}
df_results = df_results.rename(columns=rename_results)

keep_cols = ["WellName", "Constituent", "RealSampleDate", "Result", "Units"]
missing = [c for c in keep_cols if c not in df_results.columns]
if missing:
    raise KeyError(f"Missing expected result column(s): {missing}")

df_results = df_results[keep_cols].copy()

# Clean types
df_results["Result"] = pd.to_numeric(df_results["Result"], errors="coerce")
df_results["RealSampleDate"] = to_gwsdat_date(df_results["RealSampleDate"])

# Normalize WellName
df_results["WellName"] = df_results["WellName"].map(normalize_well_id)

# ----------------------
# 2) Read + process coordinate & segment data
# ----------------------
df_coord = pd.read_excel(coord_path, engine="openpyxl")
df_seg   = pd.read_excel(seg_path,   engine="openpyxl")

# Expecting these columns (adjust if your actual schema differs)
# dt_coordinate: sys_loc_code, x_coord, y_coord, coord_type_code
# dt_well_segment: sys_loc_code, start_depth, end_depth
needed_coord = {"sys_loc_code", "x_coord", "y_coord", "coord_type_code"}
needed_seg   = {"sys_loc_code", "start_depth", "end_depth"}

if not needed_coord.issubset(df_coord.columns):
    raise KeyError(f"dt_coordinate is missing columns: {sorted(needed_coord - set(df_coord.columns))}")
if not needed_seg.issubset(df_seg.columns):
    raise KeyError(f"dt_well_segment is missing columns: {sorted(needed_seg - set(df_seg.columns))}")

# Aggregate segment depths per well to a single interval (min start, max end)
seg_agg = (
    df_seg
    .groupby("sys_loc_code", as_index=False)
    .agg(start_depth=("start_depth", "min"), end_depth=("end_depth", "max"))
)

# Merge coord + aggregated segments
df_wc = pd.merge(df_coord, seg_agg, on="sys_loc_code", how="left")

# Normalize WellName
df_wc["WellName"] = df_wc["sys_loc_code"].map(normalize_well_id)

# Split by coordinate system
# NOTE: Following your original choice: for LATLONG_NAD83 rows, treat XCoord as LAT and YCoord as LONG.
latlong = (
    df_wc.loc[df_wc["coord_type_code"].eq("LATLONG_NAD83"),
              ["WellName", "x_coord", "y_coord"]]
          .rename(columns={"x_coord": "LAT", "y_coord": "LONG"})
)

sp = (
    df_wc.loc[df_wc["coord_type_code"].eq("SP_KS_S_NAD83_ft"),
              ["WellName", "x_coord", "y_coord", "start_depth", "end_depth"]]
          .rename(columns={"x_coord": "XCoord", "y_coord": "YCoord"})
)

# Combine: one row per WellName with SP coords + (optional) LAT/LONG
df_well_coord = pd.merge(sp, latlong, on="WellName", how="left")

# If some wells only have LAT/LONG and not SP, keep them too:
only_latlong = latlong.loc[~latlong["WellName"].isin(df_well_coord["WellName"])]
if not only_latlong.empty:
    only_latlong = only_latlong.assign(XCoord=np.nan, YCoord=np.nan, start_depth=np.nan, end_depth=np.nan)
    df_well_coord = pd.concat([df_well_coord, only_latlong], ignore_index=True)

# Arrange columns
df_well_coord = df_well_coord[["WellName", "XCoord", "YCoord", "LAT", "LONG", "start_depth", "end_depth"]]

# ----------------------
# 3) Intersections using CLEANED WellName
# ----------------------
wells_in_coords = set(df_well_coord["WellName"].dropna())
wells_in_results = set(df_results["WellName"].dropna())

wellsBoth = sorted(wells_in_coords & wells_in_results)
print(f"Wells in coordinate ∩ results: {len(wellsBoth)}")

# ----------------------
# 4) Active list (cleaned) and final intersection
# ----------------------
active_raw = (
    pd.read_csv(active_csv, header=None)
      .iloc[:, 0]
      .astype(str)
      .map(normalize_well_id)
      .dropna()
      .unique()
)
active_set = set(active_raw)

ActivePlusMonitored = sorted(active_set & set(wellsBoth))
print(f"Active wells with monitoring data: {len(ActivePlusMonitored)}")

# Additional wells to remove
# addl_remove = {"CMW-22D", "MW-21D", "MW-4S"}
# ActivePlusMonitored = [w for w in ActivePlusMonitored if w not in addl_remove]
# print(f"Active wells with monitoring data after removing {addl_remove}: {len(ActivePlusMonitored)}")

# ----------------------
# 5) Optional diagnostics to show what got normalized
# ----------------------
def normalization_deltas(series: pd.Series, label: str, sample=10):
    s_raw = series.astype(str)
    s_norm = s_raw.map(normalize_well_id)
    changed = s_raw[s_raw.ne(s_norm)]
    if not changed.empty:
        print(f"[{label}] Normalized {changed.size} well IDs (showing up to {sample}):")
        for old, new in zip(changed.head(sample), s_norm[changed.index].head(sample)):
            print(f"  '{old}'  ->  '{new}'")

# Show deltas for each source
normalization_deltas(df_wc["sys_loc_code"], "Coordinates")
normalization_deltas(df_results["WellName"], "Results (already normalized view)")
normalization_deltas(pd.Series(active_raw), "Active list (post-read)")
# ----------------------

Wells in coordinate ∩ results: 108
Active wells with monitoring data: 62


c:\ageglio-1\EDD Data Processor\dataprocessor\Lib\site-packages\openpyxl\packaging\custom.py:213: UserWarning: Unknown type for ContentTypeId
  warn(f"Unknown type for {prop.name}")
c:\ageglio-1\EDD Data Processor\dataprocessor\Lib\site-packages\openpyxl\packaging\custom.py:213: UserWarning: Unknown type for MediaServiceImageTags
  warn(f"Unknown type for {prop.name}")


In [2]:
# Export filtered datasets for GWSDAT
df_well_coord_filtered = df_well_coord[df_well_coord["WellName"].isin(ActivePlusMonitored)]
# Remove duplicates based on WellName, keeping the first occurrence (if there are multiple entries for the same well, only the first one will be kept)
df_well_coord_filtered = df_well_coord_filtered.drop_duplicates(subset=["WellName"], keep="first")
# Filter results to wells with coordinates and export
df_results_filtered = df_results[df_results["WellName"].isin(ActivePlusMonitored)]


In [3]:
df_well_coord[df_well_coord["WellName"].str.contains("MW-4S")]

,WellName,XCoord,YCoord,LAT,LONG,start_depth,end_depth
34,CMW-4S,1.484060e+06,1.816673e+06,-97.903616,38.050237,NaN,NaN
109,MW-4S,1.485984e+06,1.817009e+06,-97.896927,38.051125,35.0,50.0


In [4]:
# create S and D aquifer labels based on well name patterns
df_well_coord_filtered.loc[df_well_coord_filtered["WellName"].str.endswith("M"), "Aquifer"] = "D"
df_well_coord_filtered.loc[df_well_coord_filtered["WellName"].str.endswith("B"), "Aquifer"] = "D"
df_well_coord_filtered.loc[df_well_coord_filtered["WellName"].str.contains("D"), "Aquifer"] = "D"
df_well_coord_filtered.loc[df_well_coord_filtered["WellName"].str.endswith("C"), "Aquifer"] = "D"
df_well_coord_filtered.loc[df_well_coord_filtered["WellName"].str.endswith("S"), "Aquifer"] = "S"
df_well_coord_filtered.loc[df_well_coord_filtered["WellName"].str.endswith("A"), "Aquifer"] = "S"
# Add RW-3, RW-4, RW-5 to D aquifer based on your note
df_well_coord_filtered.loc[df_well_coord_filtered["WellName"].isin(["RW-3", "RW-4", "RW-5"]), "Aquifer"] = "D"
# If there are any wells that don't match the above patterns, label them as "S"

df_well_coord_filtered.Aquifer.value_counts()

Aquifer
D    35
S    27
Name: count, dtype: int64

In [5]:
# I want to create a new column called SampleDate which forces the RealSampleDate to the end of the quarter that it is in.
# So if the RealSampleDate is in January, February, or March, the SampleDate will be March 31 of that year. If the RealSampleDate is in April, May, or June, the SampleDate will be June 30 of that year. If the RealSampleDate is in July, August, or September, the SampleDate will be September 30 of that year. If the RealSampleDate is in October, November, or December, the SampleDate will be December 31 of that year.
def assign_quarter_end_date(date_str):
    date = pd.to_datetime(date_str)
    if date.month in [1, 2, 3]:
        return f"{date.year}-03-01"
    elif date.month in [4, 5, 6]:
        return f"{date.year}-06-01"
    elif date.month in [7, 8, 9]:
        return f"{date.year}-09-01"
    else:
        return f"{date.year}-12-01"
df_results_filtered["SampleDate"] = df_results_filtered["RealSampleDate"].apply(assign_quarter_end_date)
df_results_filtered_d = df_results_filtered.drop(columns=["RealSampleDate"])
df_results_filtered_d = df_results_filtered_d[["WellName", "Constituent", "SampleDate", "Result", "Units"]]

In [6]:
df_well_coord_filtered.to_csv(save_coord_csv, index=False)
df_results_filtered_d.to_csv(save_monitored_csv, index=False)

In [47]:
df_results_filtered_d[(df_results_filtered_d["WellName"].str.contains("RW-5"))&(df_results_filtered_d["SampleDate"]>="2011-01-01")]

,WellName,Constituent,SampleDate,Result,Units
2308,RW-5,Chloride,2011-03-01,840.0,mg/L
2309,RW-5,Chloride,2011-06-01,800.0,mg/L
2310,RW-5,Chloride,2011-09-01,1070.0,mg/L
2311,RW-5,Chloride,2011-12-01,1330.0,mg/L
2312,RW-5,Chloride,2012-06-01,820.0,mg/L
2313,RW-5,Chloride,2012-09-01,747.0,mg/L
2314,RW-5,Chloride,2012-12-01,690.0,mg/L
2315,RW-5,Chloride,2013-03-01,705.0,mg/L
2316,RW-5,Chloride,2013-06-01,750.0,mg/L
2317,RW-5,Chloride,2013-09-01,1010.0,mg/L
